In [ ]:
import json
import boto3
import os
import time
import urllib3
import uuid
from datetime import datetime, timezone
from urllib.parse import urlparse
from boto3.dynamodb.conditions import Key

# clients
s3 = boto3.client("s3")
ddb = boto3.resource("dynamodb")
athena = boto3.client("athena")
http = urllib3.PoolManager()

# env
BUCKET = os.environ["ATHENA_RESULT_BUCKET"]
TABLE_NAME = os.environ["USER_TABLE"]
DB = os.environ["RAW_DB"]
RAW_TABLE = os.environ["RAW_TABLE"]
GSI = os.environ["USER_TOKEN_GSI"]
API = os.environ["API_ENDPOINT"]

table = ddb.Table(TABLE_NAME)

def run_athena(sql):
    res = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DB},
        ResultConfiguration={"OutputLocation": f"s3://{BUCKET}/"}
    )

    qid = res["QueryExecutionId"]

    for _ in range(60):
        status = athena.get_query_execution(QueryExecutionId=qid)
        state = status["QueryExecution"]["Status"]["State"]
        if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
            break
        time.sleep(1)

    if state != "SUCCEEDED":
        raise Exception(f"Athena failed: {state}")

    return athena.get_query_results(QueryExecutionId=qid)

def gps_to_obj(gps):
    if not gps:
        return None
    try:
        lat, lon = gps.split(",")
        return {"lat": float(lat), "lon": float(lon)}
    except:
        return None

def get_latest_gps(tokens):
    if not tokens:
        return {}

    ids = ",".join([f"'{u}'" for u in tokens])

    sql = f"""
    SELECT
        user_token,
        max_by(coarse_gps, event_ts)
    FROM {RAW_TABLE}
    WHERE coarse_gps IS NOT NULL
      AND user_token IN ({ids})
      AND dt >= current_date - interval '1' day
    GROUP BY user_token
    """

    result = run_athena(sql)

    gps_map = {}
    rows = result["ResultSet"]["Rows"][1:]

    for r in rows:
        user = r["Data"][0]["VarCharValue"]
        gps = r["Data"][1]["VarCharValue"]
        gps_map[user] = gps

    return gps_map

def extract_key(url):
    if url.startswith("s3://"):
        return url.replace("s3://", "").split("/", 1)[1]
    parsed = urlparse(url)
    path = parsed.path.lstrip("/")
    return path.split("/", 1)[1]

def lambda_handler(event, context):
    url = event["detail"]["outputLocation"]
    key = extract_key(url)

    if not key.endswith(".csv"):
        return {"status": "skip"}

    obj = s3.get_object(Bucket=BUCKET, Key=key)
    content = obj["Body"].read().decode()

    tokens = [u.strip() for u in content.split("\n") if u.strip()]

    gps_map = get_latest_gps(tokens)

    payloads = []

    for token in tokens:
        payloads.append({
            "event_id": str(uuid.uuid4()),
            "user_token": token,
            "gps": gps_to_obj(gps_map.get(token)),
            "occurred_at": datetime.now(timezone.utc).isoformat()
        })

    for p in payloads:
        http.request(
            "POST",
            API,
            body=json.dumps(p).encode(),
            headers={"Content-Type": "application/json"}
        )

    return {"status": "success", "count": len(payloads)}
